loading dataframe from

In [ ]:
import pandas as pd

df = pd.read_csv('https://drive.google.com/uc?export=download&id=1k26vJfnpQx-Vl509-pjxhBkWu5efS12s')

In [ ]:
!jupyter nbconvert --execute --to html "/content/Project_Notebook_2_Baseline.ipynb"

[NbConvertApp] Converting notebook /content/Project_Notebook_2_Baseline.ipynb to html
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
Traceback (most recent call last):
  File "/usr/local/bin/jupyter-nbconvert", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/jupyter_core/application.py", line 284, in launch_instance
    super().launch_instance(argv=argv, **kwargs)
  File

before we can test our models, we need to build a working pipeline that produces reasonable results.
we picked [Henghes et al. 2021](https://academic.oup.com/mnras/article/505/4/4847/6288434) as our main refrence. since its almost 1-1 with what we are doing in our project. yes its an older paper but its too useful not to use.

they benchmarked 5 of the same ML models we plan to use (LR, kNN, RF, BDT ≈ XGBoost, MLP) on SDSS DR12 with the same 9 features (5 magnitudes + 4 colors).



# setup

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# define features X and target y
# our features are magnitudes + color
# the target is what we are trying to estimate, redshift.
feature_cols = ['u', 'g', 'r', 'i', 'z', 'u_g', 'g_r', 'r_i', 'i_z']
X = df[feature_cols].values
y = df['redshift'].values

print(f"{X.shape}")
print(f"{y.shape}")

(57241, 9)
(57241,)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,     # 20% for testing
    random_state=42    # reproducibility
)

print(f"Training set: {X_train.shape[0]} galaxies")
print(f"Test set: {X_test.shape[0]} galaxies")

Training set: 45792 galaxies
Test set: 11449 galaxies


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training mean: {X_train_scaled.mean(axis=0).round(3)}")
print(f"Training std:  {X_train_scaled.std(axis=0).round(3)}")

Training mean: [-0.  0. -0. -0.  0. -0. -0.  0. -0.]
Training std:  [1. 1. 1. 1. 1. 1. 1. 1. 1.]


# Section 1 - Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)

print(f"First 5 predictions: {y_pred_lr[:5].round(4)}")
print(f"First 5 true values: {y_test[:5].round(4)}")

First 5 predictions: [0.4785 0.1373 0.1737 0.5747 0.5267]
First 5 true values: [0.5204 0.0382 0.1102 0.5238 0.5556]


reasonable predictions, its fine

###evaluation

our metrics are the following;

- **MAE** (mean absolute error): average of |predicted - actual|. easy to interpret.



---


- **MSE** (mean squared error): punishes large errors.



---


- **bias**: tells us if the model systematically over or under-predicts.


---



- **σ_NMAD**: measures how tightly our predictions cluster around the true values. it's based on the median error instead of the mean, so a handful of bad predictions can't inflate it. lower = more consistent predictions. this is the standard scatter metric in photometric redshift papers.

***Important note.***

  photo-z (photometric redshift) papers don't all use the same formula. Henghes (2021) calls it "Precision" and uses the older Ilbert (2006) definition:
  
`  1.48 × median(|Δz_norm|). `
   
   Treyer (2024) uses the modern statistical NMAD:
   
` 1.48 × median(|Δz_norm - median(Δz_norm)|) `
  
  the same idea but centered around the median first. the numbers differ slightly for biased models.


  we use Henghes's formula since he's our main baseline paper. this lets us compare σ_NMAD directly against their Table 2. we still call it σ_NMAD in our code because that's the name used by most recent papers (Li 2023, Zhou 2024).

---


- **outlier fraction**: the percentage of galaxies where our prediction was very wrong.

    "very wrong" means `|z_predicted - z_actual|/(1 + z_actual) > 0.15`
    
    e.g. predicting z = 0.1 for a galaxy that's actually at z = 0.5.

  ***Important note.***

    same caveat as σ_NMAD: Henghes (2021) uses an older definition

    `(|z_pred - z_spec| > 0.10). `
    
    we use the modern normalized version that matches recent litretuare. since this a bit too outdated, but also there isnt really much of a standard on this anyway
  
   this means we can't directly compare outlier fraction to Henghes, but we can compare to the other 3 papers later on in our paper.
  



In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def evaluate(y_true, y_pred, model_name):
    # Standard regression metrics
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)

    # Photo-z specific metrics
    dz = (y_pred - y_true) / (1 + y_true)
    bias = np.mean(dz)
    sigma_nmad = 1.48 * np.median(np.abs(dz))
    outlier_fraction = np.mean(np.abs(dz) > 0.15)

    print(f"\n--- {model_name} ---")
    print(f"MAE:              {mae:.4f}")
    print(f"MSE:              {mse:.4f}")
    print(f"Bias:             {bias:.4f}")
    print(f"σ_NMAD:           {sigma_nmad:.4f}")
    print(f"Outlier fraction: {outlier_fraction:.4%}")

    return {'model': model_name, 'MAE': mae, 'MSE': mse,
            'bias': bias, 'sigma_NMAD': sigma_nmad,
            'outlier_fraction': outlier_fraction}

In [ ]:
#compare results
pd.DataFrame({
    'Ours':    [0.0712, 0.0118, 0.0054, 0.0500],
    'Henghes': [0.0509, 0.0057, 0.0397, 0.0434],
}, index=['MAE', 'MSE', 'Bias', 'σ_NMAD']).round(4)

,Ours,Henghes
MAE,0.0712,0.0509
MSE,0.0118,0.0057
Bias,0.0054,0.0397
σ_NMAD,0.0500,0.0434


our LR is worse than Henghes's on MSE/MAE but better on bias.

i think its largely because of the different sample size

since we know the relationship between our features and resshift is non-linear. it might have helped them having a smaller dataset.

what matters here is that we know that LR will preform worst among our models. it doesnt really have to be 1-1 with the paper



# Section 2 - k-Nearest Neighbors

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

knn = KNeighborsRegressor(n_neighbors=10)
knn.fit(X_train_scaled, y_train)

y_pred_knn = knn.predict(X_test_scaled)

results_knn = evaluate(y_test, y_pred_knn, "k-Nearest Neighbors")


--- k-Nearest Neighbors ---
MAE:              0.0482
MSE:              0.0080
Bias:             0.0028
σ_NMAD:           0.0274
Outlier fraction: 2.9610%


###evaluation

In [ ]:
pd.DataFrame({
    'Ours':    [0.0482, 0.0080, 0.0028, 0.0276],
    'Henghes': [0.0409, 0.0044, 0.0310, 0.0318],
}, index=['MAE', 'MSE', 'Bias', 'σ_NMAD']).round(4)

,Ours,Henghes
MAE,0.0482,0.0409
MSE,0.0080,0.0044
Bias,0.0028,0.0310
σ_NMAD,0.0276,0.0318


cour MSE is ~2× worse and MAE is slightly worse . this is the same untuned vs tuned gap we saw with LR. Henghes's optimal kNN used k=21 with distance weighting after extensive tuning. we're using k=10 uniform weighting scikit-learn defaults.

but the main baseline goal is met, kNN clearly beats LR  which is the qualitative finding Henghes reports. our pipeline ranks models correctly.

#Section 3 - Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_scaled, y_train) #100 is arbitrary for now

y_pred_rf = rf.predict(X_test_scaled)

results_rf = evaluate(y_test, y_pred_rf, "Random Forest")


--- Random Forest ---
MAE:              0.0468
MSE:              0.0076
Bias:             0.0019
σ_NMAD:           0.0260
Outlier fraction: 2.9784%


###evaluation

In [ ]:
pd.DataFrame({
    'Ours':    [0.0468, 0.0076, 0.0019, 0.0260],
    'Henghes': [0.0385, 0.0042, 0.0293, 0.0293],
}, index=['MAE', 'MSE', 'Bias', 'σ_NMAD']).round(4)

,Ours,Henghes
MAE,0.0468,0.0385
MSE,0.0076,0.0042
Bias,0.0019,0.0293
σ_NMAD,0.0260,0.0293


### RF results

the same ~2× MSE gap

this is almost certainly still the untuned-vs-tuned story. Henghes's had optimal RF hyperparameters after 1000 iterations of random search. our defaults are obvioully not optimal. proper tuning should close this gap (what we will do later)

what we can note though is that our bias is much better . Henghes's RF  over predicts redshift. idk why, probably one of the hyperparameters




# Section 3 - XGBoost

In [ ]:
from xgboost import XGBRegressor

xgb = XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1)
xgb.fit(X_train_scaled, y_train)

y_pred_xgb = xgb.predict(X_test_scaled)

results_xgb = evaluate(y_test, y_pred_xgb, "XGBoost")


--- XGBoost ---
MAE:              0.0476
MSE:              0.0078
Bias:             0.0019
σ_NMAD:           0.0267
Outlier fraction: 3.0920%


###evaluation

In [ ]:
pd.DataFrame({
    'Ours':    [0.0476, 0.0078, 0.0019, 0.0267],
    'Henghes': [0.0388, 0.0043, 0.0294, 0.0290],
}, index=['MAE', 'MSE', 'Bias', 'σ_NMAD']).round(4)

,Ours,Henghes
MAE,0.0476,0.0388
MSE,0.0078,0.0043
Bias,0.0019,0.0294
σ_NMAD,0.0267,0.0290


XGBoost is the modern version of Henghes's Boosted Decision Tree (BDT). is it a fair comparison?

basically yes, same idea diff implementation

they train data sequentially. but XGBoost is faster and has better regularation. our XGBoost is untuned though so it looks bad anyway.



#Section 4 - MLP

In [ ]:
from sklearn.neural_network import MLPRegressor

mlp = MLPRegressor(random_state=42, max_iter=200)
mlp.fit(X_train_scaled, y_train)

y_pred_mlp = mlp.predict(X_test_scaled)

results_mlp = evaluate(y_test, y_pred_mlp, "Multi-Layer Perceptron")


--- Multi-Layer Perceptron ---
MAE:              0.0503
MSE:              0.0078
Bias:             -0.0006
σ_NMAD:           0.0314
Outlier fraction: 3.0570%


###evaluation

In [ ]:
pd.DataFrame({
    'Ours':    [0.0503, 0.0078, -0.0006, 0.0314],
    'Henghes': [0.0513, 0.0047, 0.0346, 0.0408],
}, index=['MAE', 'MSE', 'Bias', 'σ_NMAD']).round(4)

,Ours,Henghes
MAE,0.0503,0.0513
MSE,0.0078,0.0047
Bias,-0.0006,0.0346
σ_NMAD,0.0314,0.0408


a bit suprised by the results. our single layer defualt MLP preformed on bar with others.

roughly same MSE as our other basline models. better MAE and bias than our ref paper

this is probably because we have more data. but could be also that the ref paper used tnah for activition, the defualt here is ReLU

#Section 6 - Summary

In [ ]:
summary = pd.DataFrame([results_lr, results_knn, results_rf, results_xgb, results_mlp])
summary = summary.set_index('model').round(4)
summary['outlier_fraction'] = (summary['outlier_fraction'] * 100).round(2).astype(str) + '%'
summary.columns = ['MAE', 'MSE', 'Bias', 'σ_NMAD', 'Outlier %']
summary

,MAE,MSE,Bias,σ_NMAD,Outlier %
model,,,,,
Linear Regression,0.0712,0.0118,0.0054,0.0500,5.34%
k-Nearest Neighbors,0.0482,0.0080,0.0028,0.0274,2.96%
Random Forest,0.0468,0.0076,0.0019,0.0260,2.98%
XGBoost,0.0476,0.0078,0.0019,0.0267,3.09%
Multi-Layer Perceptron,0.0503,0.0078,-0.0006,0.0314,3.06%



non-linear models beat linear by a lot

 LR's MSE is 1.5× worse than everything else. the relationship between magnitudes and redshift is clearly non-linear, just as the EDA predicted.


How does MSE compare with the ref paper?


| model | Henghes | ours |
|---|---|---|
| RF | 0.0042 | 0.0076 |
| XGBoost| 0.0043 | 0.0078 |
| kNN | 0.0044 | 0.0080 |
| MLP | 0.0047 | 0.0078 |
| LR | 0.0057 | 0.0118 |



rankings

| rank | Henghes | ours |
|---|---|---|
| 1 | RF | RF |
| 2 | BDT | XGBoost |
| 3 | kNN | MLP |
| 4 | MLP | kNN |
| 5 | LR | LR |


RF wins in both. LR loses in both. the boosted tree comes in second in both. the only swap is MLP vs kNN in the middle - and those two models are within 0.0002 MSE of each other in our results so its not an issue.

pipeline works, model ordering matches Henghes almost exactly. numbers land in expected photo-z ranges, and the gap to Henghes has a clear cause and solution (tuning).



---



---



In [ ]:
import joblib

joblib.dump(knn_search.best_estimator_, 'knn.joblib')